In [ ]:
# 31. Implement the CBOW model to learn word embeddings from a small text 
# corpus. Generate context and target word pairs, train the model using an 
# Embedding layer and Softmax output layer, and display the learned word 
# embeddings and predicted words.

In [1]:
# Step 1: Import required libraries
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import re

In [2]:
# --------------------------------
# a. DATA PREPARATION
# --------------------------------

# Step 2: Create sample text corpus
text = """
the black cat sits on the warm mat
the brown dog sits on the cold floor
the fluffy cat likes to chase the playful dog
the friendly dog likes to play with the cute cat
deep learning and natural language processing are fascinating
word embeddings capture semantic meanings of words
continuous bag of words model learns context efficiently
"""

In [3]:
# Step 3: Convert text into lowercase and remove punctuation
text = re.sub(r'[^\w\s]', '', text.lower())

# Tokenize text into words
words = text.split()

print("Tokenized Words:")
print(words)

Tokenized Words:
['the', 'black', 'cat', 'sits', 'on', 'the', 'warm', 'mat', 'the', 'brown', 'dog', 'sits', 'on', 'the', 'cold', 'floor', 'the', 'fluffy', 'cat', 'likes', 'to', 'chase', 'the', 'playful', 'dog', 'the', 'friendly', 'dog', 'likes', 'to', 'play', 'with', 'the', 'cute', 'cat', 'deep', 'learning', 'and', 'natural', 'language', 'processing', 'are', 'fascinating', 'word', 'embeddings', 'capture', 'semantic', 'meanings', 'of', 'words', 'continuous', 'bag', 'of', 'words', 'model', 'learns', 'context', 'efficiently']


In [4]:
# --------------------------------
# b. GENERATE TRAINING DATA
# --------------------------------

# Step 4: Create vocabulary
vocab = sorted(set(words))
vocab_size = len(vocab)

print("Vocabulary:")
print(vocab)

print("Vocabulary Size:", vocab_size)

Vocabulary:
['and', 'are', 'bag', 'black', 'brown', 'capture', 'cat', 'chase', 'cold', 'context', 'continuous', 'cute', 'deep', 'dog', 'efficiently', 'embeddings', 'fascinating', 'floor', 'fluffy', 'friendly', 'language', 'learning', 'learns', 'likes', 'mat', 'meanings', 'model', 'natural', 'of', 'on', 'play', 'playful', 'processing', 'semantic', 'sits', 'the', 'to', 'warm', 'with', 'word', 'words']
Vocabulary Size: 41


In [5]:
# Step 5: Assign unique indices to words

# Word to index mapping
word_to_index = {word: i for i, word in enumerate(vocab)}

# Index to word mapping
index_to_word = {i: word for word, i in word_to_index.items()}

print("Word to Index:")
print(word_to_index)

Word to Index:
{'and': 0, 'are': 1, 'bag': 2, 'black': 3, 'brown': 4, 'capture': 5, 'cat': 6, 'chase': 7, 'cold': 8, 'context': 9, 'continuous': 10, 'cute': 11, 'deep': 12, 'dog': 13, 'efficiently': 14, 'embeddings': 15, 'fascinating': 16, 'floor': 17, 'fluffy': 18, 'friendly': 19, 'language': 20, 'learning': 21, 'learns': 22, 'likes': 23, 'mat': 24, 'meanings': 25, 'model': 26, 'natural': 27, 'of': 28, 'on': 29, 'play': 30, 'playful': 31, 'processing': 32, 'semantic': 33, 'sits': 34, 'the': 35, 'to': 36, 'warm': 37, 'with': 38, 'word': 39, 'words': 40}


In [6]:
# Step 6: Generate context-target pairs

# Define context window size
window_size = 2

contexts = []
targets = []

# Generate context and target words
for i in range(window_size, len(words) - window_size):

    context = []

    for j in range(i - window_size, i + window_size + 1):
        if j != i:
            context.append(word_to_index[words[j]])

    contexts.append(context)
    targets.append(word_to_index[words[i]])

# Convert into NumPy arrays
contexts = np.array(contexts)
targets = np.array(targets)

print("Total Training Samples:", len(contexts))

Total Training Samples: 54


In [7]:
# Step 7: Display sample training pairs

for i in range(5):

    context_words = [index_to_word[idx] for idx in contexts[i]]

    print("Context:", context_words)
    print("Target:", index_to_word[targets[i]])
    print()

Context: ['the', 'black', 'sits', 'on']
Target: cat

Context: ['black', 'cat', 'on', 'the']
Target: sits

Context: ['cat', 'sits', 'the', 'warm']
Target: on

Context: ['sits', 'on', 'warm', 'mat']
Target: the

Context: ['on', 'the', 'mat', 'the']
Target: warm



In [8]:
# --------------------------------
# c. BUILD AND TRAIN CBOW MODEL
# --------------------------------

# Step 8: Build CBOW model
model = keras.Sequential([
    layers.Input(shape=(window_size * 2,)),

    # Embedding layer
    layers.Embedding(
        input_dim=vocab_size,
        output_dim=50
    ),

    # Average context word embeddings
    layers.GlobalAveragePooling1D(),

    # Softmax output layer
    layers.Dense(vocab_size, activation="softmax")
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 4, 50)               │           2,050 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ (None, 50)                  │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 41)                  │           2,091 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,141 (16.18 KB)

 Trainable params: 4,141 (16.18 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# Step 9: Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
# Step 10: Train the CBOW model
history = model.fit(
    contexts,
    targets,
    epochs=1000,
    verbose=0
)

print("Model Training Completed!")

Model Training Completed!


In [11]:
# --------------------------------
# d. OUTPUT AND EVALUATION
# --------------------------------

# Step 11: Evaluate the model
loss, accuracy = model.evaluate(contexts, targets, verbose=0)

print("Training Loss:", loss)
print("Training Accuracy:", accuracy)

Training Loss: 0.0005887438892386854
Training Accuracy: 1.0


In [12]:
# Step 12: Display learned word embeddings

# Extract embeddings from Embedding layer
embeddings = model.layers[0].get_weights()[0]

# Display first five word embeddings
for i in range(5):
    print(vocab[i], ":", embeddings[i])

and : [ 0.4856641   0.63067    -0.332573   -0.97473663 -0.1378037   0.856725
  0.5992458  -0.10460695  0.86836004 -0.62868756  0.7338403  -0.7100714
 -0.57263297  0.88347876  0.7626228  -1.148603   -0.01179013  0.6353674
 -1.0998236  -0.48733675 -0.4655369  -0.7924436  -1.0901477  -0.10652412
  0.43023497 -0.55433035 -0.22021909  0.57859993 -0.47866178 -0.2233677
  0.6751525  -0.03218531  1.2572188   0.7556663  -1.2607768   1.2762196
  0.95267576  0.8953262  -0.69143116 -0.65354    -0.66882026 -0.4174166
  1.1153597   0.51770455  0.7612285  -0.96352017  0.63402295  0.72686094
  0.9280442   0.7390373 ]
are : [ 0.7873561   0.7787185  -0.7406738  -0.6661238  -0.29165244 -0.4586348
 -0.8971459  -1.6554306   1.0711919  -0.01506322  0.28625157 -0.6574716
 -1.164937    1.0822763   0.44356346 -0.54341996  0.98008686 -0.97604144
 -0.51365644 -1.4573104  -0.22214992  0.9693749   0.74937034 -1.15431
  1.0546306   0.99398077 -0.34293655  0.12199534 -0.1158879   0.836161
  0.26975134  1.0943412  -0

In [13]:
# Step 13: Predict the target word

# Give sample context words
sample_context = ["the", "black", "sits", "on"]

# Convert context words into indices
sample = np.array([
    [word_to_index[word] for word in sample_context]
])

# Predict the target word
prediction = model.predict(sample, verbose=0)

predicted_index = np.argmax(prediction)

print("Given Context:", sample_context)
print("Predicted Target Word:", index_to_word[predicted_index])

Given Context: ['the', 'black', 'sits', 'on']
Predicted Target Word: cat
